In [2]:
import os
from datetime import datetime, timedelta
import pandas as pd

# ========== 配置区域 ==========
BASE_DIR = "/data/sigma/zzy/janus/results/vol_pred_prod/results"
START_DATE = "2025-01-10"
END_DATE = "2026-03-31"
SYMBOLS = [
    "XRPUSDT",
]
# ==============================

# 生成日期范围（仅工作日+周末都算，每个自然日）
start = datetime.strptime(START_DATE, "%Y-%m-%d")
end = datetime.strptime(END_DATE, "%Y-%m-%d")
all_dates = []
cur = start
while cur <= end:
    all_dates.append(cur.strftime("%Y-%m-%d"))
    cur += timedelta(days=1)

print(f"日期范围: {START_DATE} ~ {END_DATE}, 共 {len(all_dates)} 天")
print(f"检查币种: {SYMBOLS}")
print("=" * 60)

# 检查每个币种
records = []
for symbol in SYMBOLS:
    missing_dates = []
    existing_dates = []
    for date_str in all_dates:
        path = os.path.join(BASE_DIR, date_str, symbol, "results.npz")
        if os.path.exists(path):
            existing_dates.append(date_str)
        else:
            missing_dates.append(date_str)

    total = len(all_dates)
    exist_count = len(existing_dates)
    miss_count = len(missing_dates)
    pct = exist_count / total * 100 if total > 0 else 0

    records.append({
        "币种": symbol,
        "总天数": total,
        "已有": exist_count,
        "缺失": miss_count,
        "完整度": f"{pct:.1f}%",
    })

    print(f"\n【{symbol}】已有 {exist_count}/{total} 天, 完整度 {pct:.1f}%")
    if missing_dates:
        print(f"  缺失 {miss_count} 天:")
        # 将连续缺失日期合并为区间显示
        ranges = []
        range_start = missing_dates[0]
        prev = missing_dates[0]
        for d in missing_dates[1:]:
            prev_dt = datetime.strptime(prev, "%Y-%m-%d")
            cur_dt = datetime.strptime(d, "%Y-%m-%d")
            if (cur_dt - prev_dt).days == 1:
                prev = d
            else:
                ranges.append((range_start, prev))
                range_start = d
                prev = d
        ranges.append((range_start, prev))

        for rs, re in ranges:
            if rs == re:
                print(f"    {rs}")
            else:
                days = (datetime.strptime(re, "%Y-%m-%d") - datetime.strptime(rs, "%Y-%m-%d")).days + 1
                print(f"    {rs} ~ {re} ({days}天)")
    else:
        print("  ✓ 数据完整，无缺失")

print("\n" + "=" * 60)
df_summary = pd.DataFrame(records)
display(df_summary)

日期范围: 2025-01-10 ~ 2026-03-31, 共 446 天
检查币种: ['XRPUSDT']

【XRPUSDT】已有 446/446 天, 完整度 100.0%
  ✓ 数据完整，无缺失



,币种,总天数,已有,缺失,完整度
0,XRPUSDT,446,446,0,100.0%
